In [1]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("Imports successful!")
print("PyTorch:", torch.__version__)

Imports successful!
PyTorch: 2.13.0+cpu


In [2]:
# Load processed datasets

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/validation.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nColumns:")
print(train_df.columns)

Train: (1728, 3)
Validation: (370, 3)
Test: (371, 3)

Columns:
Index(['file_content', 'document_type', 'label'], dtype='str')


In [3]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

d:\DocuTune\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 30522


In [5]:
def tokenize_function(batch):
    return tokenizer(
        batch["file_content"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

In [6]:
train_encodings = tokenize_function(train_df.to_dict("list"))
val_encodings = tokenize_function(val_df.to_dict("list"))
test_encodings = tokenize_function(test_df.to_dict("list"))

print("Tokenization complete!")
print("Training examples:", len(train_encodings["input_ids"]))
print("Tokens per example:", len(train_encodings["input_ids"][0]))

Tokenization complete!
Training examples: 1728
Tokens per example: 256


In [7]:
import torch
from torch.utils.data import Dataset


class DocumentDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, index):
        item = {
            key: torch.tensor(value[index])
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[index])

        return item

    def __len__(self):
        return len(self.labels)

In [8]:
train_dataset = DocumentDataset(
    train_encodings,
    train_df["label"]
)

val_dataset = DocumentDataset(
    val_encodings,
    val_df["label"]
)

test_dataset = DocumentDataset(
    test_encodings,
    test_df["label"]
)

print("PyTorch datasets created!")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

PyTorch datasets created!
Train: 1728
Validation: 370
Test: 371


In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={
        0: "Invoices",
        1: "Purchase Orders",
        2: "Shipping Orders"
    },
    label2id={
        "Invoices": 0,
        "Purchase Orders": 1,
        "Shipping Orders": 2
    }
)

print("Baseline model loaded!")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Baseline model loaded!


In [10]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.2f}%"
)

Total parameters: 66,955,779
Trainable parameters: 66,955,779
Trainable percentage: 100.00%


In [11]:
from sklearn.metrics import accuracy_score, f1_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }

In [12]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../results/baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_dir="../results/logs",
    logging_steps=50,
    report_to="none"
)

print("Training configuration ready!")

Training configuration ready!


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Trainer ready!")

Trainer ready!


In [14]:
train_result = trainer.train()

  0%|          | 0/648 [00:00<?, ?it/s]

d:\DocuTune\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': 0.3955, 'grad_norm': 0.3864882290363312, 'learning_rate': 1.8456790123456792e-05, 'epoch': 0.23}
{'loss': 0.0177, 'grad_norm': 0.13753098249435425, 'learning_rate': 1.6913580246913582e-05, 'epoch': 0.46}
{'loss': 0.0075, 'grad_norm': 0.06565739214420319, 'learning_rate': 1.537037037037037e-05, 'epoch': 0.69}
{'loss': 0.0047, 'grad_norm': 0.05231580510735512, 'learning_rate': 1.3827160493827162e-05, 'epoch': 0.93}


  0%|          | 0/47 [00:00<?, ?it/s]

{'eval_loss': 0.0025728687178343534, 'eval_accuracy': 1.0, 'eval_macro_f1': 1.0, 'eval_runtime': 59.9733, 'eval_samples_per_second': 6.169, 'eval_steps_per_second': 0.784, 'epoch': 1.0}


...

d:\DocuTune\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': 0.0015, 'grad_norm': 0.024384144693613052, 'learning_rate': 6.111111111111112e-06, 'epoch': 2.08}
{'loss': 0.0013, 'grad_norm': 0.020561853423714638, 'learning_rate': 4.567901234567902e-06, 'epoch': 2.31}
{'loss': 0.0013, 'grad_norm': 0.022217607125639915, 'learning_rate': 3.0246913580246917e-06, 'epoch': 2.55}
{'loss': 0.0012, 'grad_norm': 0.0178063977509737, 'learning_rate': 1.4814814814814815e-06, 'epoch': 2.78}


d:\DocuTune\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/47 [00:00<?, ?it/s]

{'eval_loss': 0.0008572359220124781, 'eval_accuracy': 1.0, 'eval_macro_f1': 1.0, 'eval_runtime': 60.7388, 'eval_samples_per_second': 6.092, 'eval_steps_per_second': 0.774, 'epoch': 3.0}
{'train_runtime': 4676.2522, 'train_samples_per_second': 1.109, 'train_steps_per_second': 0.139, 'train_loss': 0.03405648576279665, 'epoch': 3.0}


In [15]:
test_results = trainer.evaluate(test_dataset)

print("Test Results:")
for key, value in test_results.items():
    print(f"{key}: {value}")

d:\DocuTune\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/47 [00:00<?, ?it/s]

Test Results:
eval_loss: 0.0025710375048220158
eval_accuracy: 1.0
eval_macro_f1: 1.0
eval_runtime: 44.6927
eval_samples_per_second: 8.301
eval_steps_per_second: 1.052
epoch: 3.0


In [16]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

predictions = trainer.predict(test_dataset)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=1
)

true_labels = test_df["label"].to_numpy()

print(classification_report(
    true_labels,
    predicted_labels,
    target_names=[
        "Invoices",
        "Purchase Orders",
        "Shipping Orders"
    ]
))

print("Confusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))

d:\DocuTune\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/47 [00:00<?, ?it/s]

                 precision    recall  f1-score   support

       Invoices       1.00      1.00      1.00       125
Purchase Orders       1.00      1.00      1.00       125
Shipping Orders       1.00      1.00      1.00       121

       accuracy                           1.00       371
      macro avg       1.00      1.00      1.00       371
   weighted avg       1.00      1.00      1.00       371

Confusion Matrix:
[[125   0   0]
 [  0 125   0]
 [  0   0 121]]


In [17]:
import json

baseline_results = {
    "model": "distilbert-base-uncased",
    "training_method": "Full Fine-Tuning",
    "total_parameters": 66955779,
    "trainable_parameters": 66955779,
    "trainable_percentage": 100.0,
    "train_samples": 1728,
    "validation_samples": 370,
    "test_samples": 371,
    "max_length": 256,
    "epochs": 3,
    "batch_size": 8,
    "learning_rate": 2e-5,
    "test_accuracy": 1.0,
    "test_macro_f1": 1.0,
    "training_time_seconds": 4676.25
}

with open("../results/baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=4)

print("Baseline results saved!")

Baseline results saved!


In [18]:
from pathlib import Path

print(Path("../results/baseline_results.json").exists())

True
